# GEN-KNOB Tuner — Multi-Adapter Fine-Tuning (E2ETune / Mistral-7B)

This notebook adopts the Multi-Adapter approach (training two independent standard LoRA adapters for PostgreSQL and MySQL) applied correctly to the `springhxm/E2ETune` base model (which is based on Mistral-7B).

It ensures the ChatML prompts from Qwen are replaced with Mistral's expected `[INST] ... [/INST]` format, while wrapping the training logic into sequential adapter tuning.

In [ ]:
!pip install -U transformers datasets peft bitsandbytes accelerate scikit-learn trl sentencepiece protobuf tiktoken huggingface_hub python-dotenv

In [ ]:
import os, json, re, math, torch
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model
from trl import SFTTrainer
from huggingface_hub import login
from dotenv import load_dotenv

load_dotenv()
torch.manual_seed(42)
np.random.seed(42)
print("Environment and imports ready.")

# 1. Configuration

In [ ]:
MODEL_ID       = "springhxm/E2ETune" 
OUTPUT_DIR_PG  = "e2etune_ma_lora_pg"
OUTPUT_DIR_MY  = "e2etune_ma_lora_mysql"
BASE_DIR       = "/home/E2ETune-AI4DB"

N_BINS         = 10
MAX_SEQ_LENGTH = 4096

DB_FILES = [
    os.path.join(BASE_DIR, "llm_tuning/postgres_combined_data.json"),
    os.path.join(BASE_DIR, "llm_tuning/mysql_combined_data.json"),
]

PGCONFIG      = os.path.join(BASE_DIR, "knob_config.json")
MYSQL32CONFIG = os.path.join(BASE_DIR, "mysql_knob_config.json")
MYSQL64CONFIG = os.path.join(BASE_DIR, "mysql64_knob_config.json")

# 2. Loading and Structuring Data

In [ ]:
print("Loading Cross-DB Data...")
all_data = []
for fpath in DB_FILES:
    if os.path.exists(fpath):
        with open(fpath, 'r') as f:
            all_data.extend(json.load(f))
    else:
        print(f"Warning: {fpath} not found.")

df = pd.DataFrame(all_data)
print(f"Total workload samples: {len(df)}")
if 'database' in df.columns:
    print(df['database'].value_counts())

# 3. Discretization

In [ ]:
with open(PGCONFIG,      'r') as f: pg_config     = json.load(f)
with open(MYSQL32CONFIG, 'r') as f: mysql32_config = json.load(f)
with open(MYSQL64CONFIG, 'r') as f: mysql64_config = json.load(f)

bucket_labels = [f"{i*10}% to {(i+1)*10}%" for i in range(10)]

def discretize_config(df_param):
    binned_configs = []
    for _, row in df_param.iterrows():
        db_name  = str(row.get('database', '')).lower()
        hw_spec  = str(row.get('hardware_specs', '')).lower()
        
        best_cfg_raw = row.get('best_config', {})
        if isinstance(best_cfg_raw, dict):
            best_cfg = best_cfg_raw.get('configuration', best_cfg_raw.get('best_config', best_cfg_raw))
        else:
            best_cfg = best_cfg_raw

        if 'postgresql' in db_name:
            schema = pg_config
        elif 'mysql' in db_name:
            if '64gb' in hw_spec:
                schema = mysql64_config
            else:
                schema = mysql32_config
        else:
            schema = {}

        binned = {}
        if isinstance(best_cfg, dict):
            for col, val in best_cfg.items():
                if pd.isna(val) or col not in schema:
                    continue
                spec = schema[col]
                if 'min' not in spec or 'max' not in spec:
                    continue
                c_min, c_max = spec['min'], spec['max']
                if c_max <= c_min:
                    binned[col] = bucket_labels[0]
                else:
                    scale_val = (max(c_min, min(c_max, float(val))) - c_min) / (c_max - c_min)
                    binned[col] = bucket_labels[min(int(scale_val * 10), 9)]
        binned_configs.append(binned)
    return binned_configs

print("Discretising knobs into textual buckets...")
df['binned_config'] = discretize_config(df)
print("Done.")

# 4. Prompt Engineering (Mistral Format)

Three improvements over the baseline prompt:

### 4a. Hardware Spec Simplification
Only `RAM`, `CPU cores`, and `CPU threads` are extracted from the raw hardware label.
The verbose machine-code identifiers (e.g. `hetzner-4c-8t-32gb`) are dropped.

### 4b. Query Plan Structured Encoding
Each query plan is flattened into a single nested-parentheses string.
If the same operator appears more than once (e.g. two `Seq Scan` nodes), their costs
are averaged and the operator appears **once** in the output — keeping the representation
compact and unambiguous.

Format: `Operator(cost=X)(Child1)(Child2)...`

### 4c. Human-Readable Metric Scaling
Internal database / OS metric values are converted to scale-annotated strings so the
language model can reason about their relative magnitude without being confused by
long digit strings:

| Raw value      | Encoded string  |
|---------------|-----------------|
| 83,438,203     | 83.4 million    |
| 1,500,000,000  | 1.5 billion     |
| 420            | 420             |
| 0.00042        | 0.00042         |

In [ ]:
# ── 4a. Hardware spec parser ──────────────────────────────────────────────────
def parse_hardware_spec(hw_raw: str) -> str:
    """
    Extract RAM (GB), CPU cores, and CPU threads from a raw hardware label.
    Handles patterns like:
        hetzner-4c-8t-32gb   →  32 GB RAM, 4 cores, 8 threads
        aws-r6i.2xlarge-64gb →  64 GB RAM  (threads/cores estimated from label)
    Falls back gracefully if the pattern is non-standard.
    """
    hw = str(hw_raw).lower()

    ram_match     = re.search(r"(\d+)\s*gb", hw)
    cores_match   = re.search(r"(\d+)\s*c(?:ore|pu)?(?:\b|[-_])", hw)
    threads_match = re.search(r"(\d+)\s*t(?:hread)?(?:\b|[-_])", hw)

    ram     = f"{ram_match.group(1)} GB"     if ram_match     else "unknown RAM"
    cores   = f"{cores_match.group(1)} cores"   if cores_match   else "unknown cores"
    threads = f"{threads_match.group(1)} threads" if threads_match else "unknown threads"

    return f"{ram} RAM, {cores}, {threads}"


# ── 4b. Query-plan encoder ────────────────────────────────────────────────────
def _extract_operator(node: dict) -> str:
    return node.get("Node Type", node.get("node_type", "Unknown"))

def _extract_cost(node: dict) -> float:
    return float(node.get("Total Cost", node.get("total_cost", node.get("cost", 0.0))))

def _flatten_plan_tree(node, operator_costs: dict):
    op   = _extract_operator(node)
    cost = _extract_cost(node)
    if op not in operator_costs:
        operator_costs[op] = [0.0, 0]
    operator_costs[op][0] += cost
    operator_costs[op][1] += 1

    for child_key in ("Plans", "plans", "children"):
        for child in node.get(child_key, []):
            _flatten_plan_tree(child, operator_costs)

def _parenthesise_str_plan(plan_str: str) -> str:
    pattern = re.compile(r"([A-Za-z][A-Za-z ]*?)\(cost=([\d.]+)\)")
    operator_costs: dict = {}
    for op, cost_str in pattern.findall(plan_str):
        op   = op.strip()
        cost = float(cost_str)
        if op not in operator_costs:
            operator_costs[op] = [0.0, 0]
        operator_costs[op][0] += cost
        operator_costs[op][1] += 1
    return _encode_operator_costs(operator_costs)

def _encode_operator_costs(operator_costs: dict) -> str:
    if not operator_costs:
        return "No plan"
    sorted_ops = sorted(operator_costs.items(), key=lambda kv: kv[1][0] / max(kv[1][1], 1), reverse=True)
    parts = []
    for op, (total, count) in sorted_ops:
        avg = total / max(count, 1)
        parts.append(f"{op}(cost={avg:.1f})")
    result = parts[0]
    for p in parts[1:]:
        result = f"{result}({p})"
    return result

def encode_query_plans(plans_raw) -> str:
    if not plans_raw:
        return "No query plans available."
    encoded = []
    for plan in plans_raw:
        if isinstance(plan, dict):
            op_costs: dict = {}
            _flatten_plan_tree(plan, op_costs)
            encoded.append(_encode_operator_costs(op_costs))
        elif isinstance(plan, str) and plan.strip():
            encoded.append(_parenthesise_str_plan(plan.strip()))
    return " | ".join(encoded) if encoded else "No query plans available."


# ── 4c. Human-readable metric scaler ─────────────────────────────────────────
def humanize_number(val) -> str:
    try:
        n = float(val)
    except (TypeError, ValueError):
        return str(val)

    abs_n = abs(n)
    sign  = "-" if n < 0 else ""

    if abs_n >= 1_000_000_000:
        return f"{sign}{abs_n / 1_000_000_000:.1f} billion"
    elif abs_n >= 1_000_000:
        return f"{sign}{abs_n / 1_000_000:.1f} million"
    elif abs_n >= 1_000:
        return f"{sign}{abs_n / 1_000:.1f} thousand"
    elif abs_n == 0:
        return "0"
    else:
        return f"{sign}{abs_n:.4g}"

def humanize_metrics(metrics: dict) -> dict:
    return {k: humanize_number(v) for k, v in metrics.items() if v != 0}


# ── Master prompt formatter ────────────────────────────────────────────────────
def format_mistral_prompt_ma(row):
    """
    Constructs a flattened Mistral-style sequence (`[INST] ... [/INST]`)
    which matches how E2ETune was originally trained and expects instructions.
    """
    db_name   = str(row.get('database', 'UNKNOWN')).upper()
    hw_parsed = parse_hardware_spec(row.get('hardware_specs', ''))

    # Internal DB/OS metrics — humanised
    raw_metrics  = row.get('internal_metrics', {}) or {}
    metrics_str  = ", ".join(f"{k} = {v}" for k, v in humanize_metrics(raw_metrics).items())

    # Workload features
    features     = row.get('workload_features', {}) or {}
    features_str = ", ".join(f"{k} = {v}" for k, v in features.items())

    # Query plans
    q_plan_str = encode_query_plans(row.get('query_plans', []))
    
    instruction = (
        f"You are an expert {db_name} Database Administrator tuning a server running on {hw_parsed} hardware. "
        "Recommend the optimal discrete bucket configurations for the given database metrics and workload properties.\n\n"
        f"WORKLOAD FEATURES: {features_str}\n"
        f"INTERNAL SYSTEM METRICS: {metrics_str}\n"
        f"QUERY PLANS: {q_plan_str}"
    )
    
    target_json = json.dumps(row['binned_config'], indent=2)
    
    # Mistral-Instruct Format
    text = f"<s>[INST] {instruction} [/INST]\n{target_json}</s>"
    return text

print("Applying Prompt Formatter...")
df['text'] = df.apply(format_mistral_prompt_ma, axis=1)

# Ensure db_type is cleanly extracted and datasets are separated
df['db_type'] = df['database'].str.lower()
pg_df = df[df['db_type'] == 'postgresql'].copy()
mysql_df = df[df['db_type'] == 'mysql'].copy()

print(f"Total PostgreSQL samples: {len(pg_df)}")
print(f"Total MySQL samples: {len(mysql_df)}")

# Train/test splits
pg_train_df, pg_test_df = train_test_split(pg_df, test_size=0.1, random_state=42)
pg_train_dataset = Dataset.from_pandas(pg_train_df[['text']])
pg_test_dataset  = Dataset.from_pandas(pg_test_df[['text']])

mysql_train_df, mysql_test_df = train_test_split(mysql_df, test_size=0.1, random_state=42)
mysql_train_dataset = Dataset.from_pandas(mysql_train_df[['text']])
mysql_test_dataset  = Dataset.from_pandas(mysql_test_df[['text']])

# 5. Model Loading & Multi-Adapter Setup

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right" # Important for Mistral

def get_base_model():
    print(f"Loading Base Model: {MODEL_ID}...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        use_cache=True,
        use_safetensors=True,
    )
    model.gradient_checkpointing_enable()
    model = prepare_model_for_kbit_training(model)
    return model

peft_config = LoraConfig(
    r=64,
    lora_alpha=128,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# 6. Sequential Multi-Adapter Training

In [ ]:
import gc

def train_and_save_adapter(train_ds, test_ds, adapter_dir):
    print(f"\n[{adapter_dir}] Initializing Train loop...\n")
    model = get_base_model()
    model = get_peft_model(model, peft_config)
    model.print_trainable_parameters()

    training_args = TrainingArguments(
        output_dir=adapter_dir,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        learning_rate=2e-5,
        num_train_epochs=4,
        fp16=True,
        logging_steps=10,
        save_strategy="epoch",
        eval_strategy="epoch",
        save_total_limit=2,
        optim="paged_adamw_32bit",
        lr_scheduler_type="cosine",
        report_to="none"
    )

    trainer = SFTTrainer(
        model=model,
        train_dataset=train_ds,
        eval_dataset=test_ds,
        peft_config=peft_config,
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        tokenizer=tokenizer,
        args=training_args,
    )

    print(f"\n[{adapter_dir}] Commencing Training...\n")
    trainer.train()

    print(f"\n[{adapter_dir}] Saving out Lora Adapter Configuration...\n")
    model.save_pretrained(adapter_dir)
    tokenizer.save_pretrained(adapter_dir)

    del model, trainer
    gc.collect()
    torch.cuda.empty_cache()


print("==================== MYSQL TUNING ====================")
train_and_save_adapter(mysql_train_dataset, mysql_test_dataset, OUTPUT_DIR_MY)

# print("================== POSTGRESQL TUNING =================")
# train_and_save_adapter(pg_train_dataset, pg_test_dataset, OUTPUT_DIR_PG)

# 7. Exporting to HuggingFace

In [ ]:
HF_REPO_NAME_PG = "NisithDissanayake/daktuner-pg-adapter"
HF_REPO_NAME_MY = "NisithDissanayake/daktuner-mysql-adapter"

def export_adapter(adapter_dir, repo_name):
    print(f"Loading {adapter_dir} + Uploading Adapter {repo_name}...")
    try:
        from peft import PeftModel
        base_model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            device_map="auto",
            dtype=torch.bfloat16
        )
        peft = PeftModel.from_pretrained(base_model, adapter_dir)

        peft.push_to_hub(repo_name)
        tokenizer.push_to_hub(repo_name)
        print(f"[{repo_name}] Successfully pushed.")

        del base_model, peft
        gc.collect()
        torch.cuda.empty_cache()
    except Exception as e:
        print(f"Hub push failed for {repo_name}: {e}")

export_adapter(OUTPUT_DIR_MY, HF_REPO_NAME_MY)
# export_adapter(OUTPUT_DIR_PG, HF_REPO_NAME_PG)

print("Done. Proceed to inference / evaluation.")